# 🎙️ Unified Batch Voice Cloning (F5-TTS & OpenVoice V2) - Google Colab T4

Notebook gabungan ini digunakan untuk memproses **F5-TTS** dan **OpenVoice V2** secara otomatis di Google Colab T4.

### 📌 Fitur Utama:
1. **Zero External Script Risk**: Tanpa `%%writefile` yang berisiko IndentationError.
2. **All Dependencies Pre-Installed**: Bebas error `ModuleNotFoundError` (`cn2an`, `pykakasi`, `fugashi`, `anyascii`, `jamo`, dll).
3. **Smart Resume**: Menyeberangi (skip) audio yang sudah pernah selesai di-generate di Drive.
4. **Direct Drive Sync**: Hasil otomatis tersimpan rapi di `/content/drive/MyDrive/data skirpsi/VoiceCloning/`.

In [ ]:
# STEP 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# STEP 2: Install System Packages & Dependensi (F5-TTS & OpenVoice)
!apt-get update -q
!apt-get install -y -q mecab libmecab-dev mecab-ipadic-utf8

!pip install --upgrade pip setuptools wheel
!pip install Cython numpy
!pip install --no-build-isolation f5-tts ffmpeg-python soundfile pydub mecab-python3 unidic num2words pythainlp subword_nmt cached_path cn2an pypinyin pykakasi g2p_en g2pM fugashi ipadic unidic_lite anyascii jamo huggingface_hub
!python -m unidic download

# Install MeloTTS & OpenVoice
!pip install --no-deps git+https://github.com/myshell-ai/MeloTTS.git
%cd /content
!rm -rf OpenVoice
!git clone --depth 1 https://github.com/myshell-ai/OpenVoice.git
%cd /content/OpenVoice
!pip install --no-deps -e .


In [ ]:
# STEP 3: Download OpenVoice V2 Checkpoints
import os, urllib.request, zipfile
from pathlib import Path

repo = Path('/content/OpenVoice')
ckpt_converter = repo / 'checkpoints_v2' / 'converter'
ckpt_base = repo / 'checkpoints_v2' / 'base_speakers'

if not ckpt_converter.exists() or not ckpt_base.exists():
    print('📥 Downloading OpenVoice V2 Checkpoints from Hugging Face Mirror...')
    zip_path = repo / 'checkpoints_v2_0417.zip'
    url = 'https://huggingface.co/cqchangm/openvoice2/resolve/main/checkpoints_v2_0417.zip'
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(repo)
    zip_path.unlink(missing_ok=True)
    print('✅ Checkpoints OpenVoice V2 berhasil diekstrak.')
else:
    print('✅ Checkpoints OpenVoice V2 sudah siap.')


In [ ]:
# STEP 4: Konfigurasi Batch
MODELS_TO_RUN = ['F5-TTS', 'OpenVoice']  # Pilih model yang ingin dijalankan
SKIP_EXISTING = True                      # Set True untuk melewatin audio yang sudah ada di Drive
BATCH_START = 0                           # Indeks awal speaker
BATCH_COUNT = 100                         # Jumlah maksimal speaker yang diproses

TEXTS = {
    'id': 'Indonesia adalah negara kepulauan yang kaya budaya, bahasa, sumber daya alam, serta memiliki masyarakat beragam yang hidup saling berdampingan harmonis.',
    'en': 'England is a historic European country known for its royal family, famous universities, football, diverse culture, and remarkable technological development worldwide.'
}
print('⚙️ Konfigurasi Batch Siap.')


In [ ]:
# STEP 5: Run Unified Batch Generation Engine
import os, sys, gc, shutil, subprocess
from pathlib import Path
import torch
import nltk

ROOT = Path('/content')
REF_ROOT = ROOT / 'reference_dataskripsi'
BASE_DRIVE_OUT = Path('/content/drive/MyDrive/data skirpsi/VoiceCloning')
REF_ROOT.mkdir(parents=True, exist_ok=True)
BASE_DRIVE_OUT.mkdir(parents=True, exist_ok=True)

def run_cmd(*args):
    cmd = [str(a) for a in args]
    return subprocess.run(cmd, capture_output=True, text=True)

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def scan_references():
    drive_folder = Path('/content/drive/MyDrive/Dataskripsi')
    if not drive_folder.exists():
        cands = list(Path('/content/drive/MyDrive').rglob('*Dataskripsi*'))
        if cands: drive_folder = cands[0]
        else: raise FileNotFoundError('Folder Dataskripsi tidak ditemukan di Google Drive!')
    
    print(f'📂 Scanning folder reference dari: {drive_folder}')
    audio_exts = {'.wav', '.mp3', '.m4a', '.ogg', '.flac'}
    copied = 0
    for f in drive_folder.rglob('*'):
        if f.is_file() and f.suffix.lower() in audio_exts:
            target_name = f.stem + '.wav' if f.suffix.lower() != '.wav' else f.name
            target_path = REF_ROOT / target_name
            if not target_path.exists():
                if f.suffix.lower() != '.wav':
                    run_cmd('ffmpeg', '-y', '-i', str(f), '-ar', '24000', '-ac', '1', str(target_path))
                else:
                    shutil.copy2(f, target_path)
                copied += 1
    print(f'  ✅ Prepared {copied} new audio file(s) di {REF_ROOT}')
    
    speaker_map = {}
    for p in REF_ROOT.glob('*.wav'):
        name = p.stem
        spk_id = name
        lang = 'id'
        if name.startswith('Suaraindo_'):
            spk_id = name[len('Suaraindo_'):]
            lang = 'id'
        elif name.startswith('Suaraeng_'):
            spk_id = name[len('Suaraeng_'):]
            lang = 'en'
        elif '_' in name:
            spk_id = name.split('_')[-1]
        
        if spk_id not in speaker_map: speaker_map[spk_id] = {'id': p, 'en': p}
        if lang == 'id': speaker_map[spk_id]['id'] = p
        elif lang == 'en': speaker_map[spk_id]['en'] = p
    return speaker_map

spk_map = scan_references()
all_speakers = sorted(list(spk_map.keys()))
selected_speakers = all_speakers[BATCH_START : BATCH_START + BATCH_COUNT]
print(f'🎙️ Total Speaker Terdeteksi: {len(all_speakers)} | Memproses Batch ({len(selected_speakers)} speaker): {selected_speakers}')

# ============================================================
# 1. MODEL F5-TTS
# ============================================================
if 'F5-TTS' in MODELS_TO_RUN:
    print('\n==================== MODEL: F5-TTS ====================')
    f5_out = ROOT / 'generated_voice_cloning' / 'F5-TTS'
    f5_drive = BASE_DRIVE_OUT / 'F5-TTS'
    f5_out.mkdir(parents=True, exist_ok=True)
    f5_drive.mkdir(parents=True, exist_ok=True)
    
    for spk in selected_speakers:
        ref_id = spk_map[spk]['id']
        ref_en = spk_map[spk]['en']
        for lang, text, ref in (('id', TEXTS['id'], ref_id), ('en', TEXTS['en'], ref_en)):
            final_mp3_name = f'{spk}_{lang}.mp3'
            drive_mp3 = f5_drive / final_mp3_name
            if SKIP_EXISTING and drive_mp3.exists() and drive_mp3.stat().st_size > 1000:
                print(f'  ⏩ [SKIP] F5-TTS {spk}_{lang} (Sudah ada di Drive)')
                continue
            
            print(f'  ⚡ [GENERATE] F5-TTS {spk}_{lang}...')
            wav_out = f5_out / f'{spk}_{lang}.wav'
            mp3_out = f5_out / final_mp3_name
            try:
                res = run_cmd('f5-tts_infer-cli', '--model', 'F5TTS_v1_Base', '--ref_audio', str(ref), '--ref_text', '', '--gen_text', text, '--output_dir', str(f5_out), '--output_file', f'{spk}_{lang}.wav')
                if wav_out.exists():
                    run_cmd('ffmpeg', '-y', '-i', str(wav_out), '-b:a', '192k', str(mp3_out))
                    shutil.copy2(mp3_out, drive_mp3)
                    print(f'  ☁️ Saved to Drive: {final_mp3_name}')
                else:
                    print(f'  ❌ [ERROR] Output WAV tidak terbentuk: {res.stderr[:200]}')
            except Exception as e:
                print(f'  ❌ [ERROR] F5-TTS {spk}_{lang}: {e}')
            finally:
                free_memory()
    free_memory()

# ============================================================
# 2. MODEL OPENVOICE V2
# ============================================================
if 'OpenVoice' in MODELS_TO_RUN:
    print('\n==================== MODEL: OpenVoice V2 ====================')
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('cmudict', quiet=True)
    
    from melo.api import TTS
    from openvoice import se_extractor
    from openvoice.api import ToneColorConverter
    
    ov_out = ROOT / 'generated_voice_cloning' / 'OpenVoice'
    ov_drive = BASE_DRIVE_OUT / 'OpenVoice'
    ov_out.mkdir(parents=True, exist_ok=True)
    ov_drive.mkdir(parents=True, exist_ok=True)
    
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    repo = Path('/content/OpenVoice')
    converter_dir = repo / 'checkpoints_v2' / 'converter'
    converter = ToneColorConverter(str(converter_dir / 'config.json'), device=device)
    converter.load_ckpt(str(converter_dir / 'checkpoint.pth'))
    
    base_tts = TTS(language='EN', device=device)
    speaker_id = next(iter(base_tts.hps.data.spk2id.values()))
    source_se = torch.load(str(repo / 'checkpoints_v2' / 'base_speakers' / 'ses' / 'en-us.pth'), map_location=device)
    
    for spk in selected_speakers:
        ref_id = spk_map[spk]['id']
        ref_en = spk_map[spk]['en']
        try:
            target_id, _ = se_extractor.get_se(str(ref_id), converter, vad=True, target_dir=str(ROOT / 'processed'))
            target_en, _ = se_extractor.get_se(str(ref_en), converter, vad=True, target_dir=str(ROOT / 'processed'))
            
            for lang, text, target in (('id', TEXTS['id'], target_id), ('en', TEXTS['en'], target_en)):
                final_mp3_name = f'{spk}_{lang}.mp3'
                drive_mp3 = ov_drive / final_mp3_name
                if SKIP_EXISTING and drive_mp3.exists() and drive_mp3.stat().st_size > 1000:
                    print(f'  ⏩ [SKIP] OpenVoice {spk}_{lang} (Sudah ada di Drive)')
                    continue
                
                print(f'  ⚡ [GENERATE] OpenVoice {spk}_{lang}...')
                tmp_wav = ov_out / f'tmp_{spk}_{lang}.wav'
                final_wav = ov_out / f'{spk}_{lang}.wav'
                final_mp3 = ov_out / final_mp3_name
                
                base_tts.tts_to_file(text, speaker_id, str(tmp_wav), speed=1.0)
                converter.convert(str(tmp_wav), source_se, target, str(final_wav), message='@Research')
                run_cmd('ffmpeg', '-y', '-i', str(final_wav), '-b:a', '192k', str(final_mp3))
                shutil.copy2(final_mp3, drive_mp3)
                print(f'  ☁️ Saved to Drive: {final_mp3_name}')
                if tmp_wav.exists(): tmp_wav.unlink()
        except Exception as e:
            print(f'  ❌ [ERROR] OpenVoice {spk}: {e}')
        finally:
            free_memory()
    
    del converter, base_tts, source_se
    free_memory()

print('\n🎉 BATCH GENERATION SELESAI UNTUK SEMUA MODEL!')
